In [1]:
import json

def get_current_weather(location, unit="fahrenheit"):
    if "seoul" in location.lower():
        return json.dumps({"location": "Seoul", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps({"lcoation": "San Francisco", "temperature": "72", "unit": unit})
    elif "paris" in location.lower():
        return json.dumps({"lcoation": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"lcoation": location, "temperature": "unknown"})

In [2]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "the city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
                },
                "required": ["location"],
            }
        }
    }
]

In [3]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv("../.env")

client = OpenAI()

messages = [
    {"role": "user", "content": "서울 날씨는 어떤가요?"}
]

response = client.chat.completions.create(
    model="gpt-6-luna",
    messages=messages,
    tools=tools,
    reasoning_effort="none"
)
print(response.to_json(indent=2))

{
  "id": "chatcmpl-Ed9f8q7PT5hFXdamzh0usya91oAmQ",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "tool_calls": [
          {
            "id": "call_m4ESQiAzBF6ezILzJSXTTehI",
            "function": {
              "arguments": "{\"location\":\"Seoul\",\"unit\":\"celsius\"}",
              "name": "get_current_weather"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1793000642,
  "model": "gpt-6-luna",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 24,
    "prompt_tokens": 161,
    "total_tokens": 185,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_de

In [ ]:
# Responses API
tools = [
    {
        "type": "function",
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and country, e.g. Seoul, South Korea",
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                },
            },
            "required": ["location"],
        },
        "strict": False,
    }
]

responses = client.responses.create(
    model="gpt-6-luna",
    input="서울 날씨는 어떤가요?",
    tools=tools,
)
for item in responses.output:
    if item.type == "function_call":
        print(f"함수 이름: {item.name}")
        print(f"전달할 인수: {item.arguments}")
        print(f"호출 ID: {item.call_id}")

함수 이름: get_current_weather
전달할 인수: {"location":"Seoul, South Korea","unit":"celsius"}
호출 ID: call_GIqXKwxfDj2qPKbLITdyZkKx


In [4]:
response_message = response.choices[0].message
messages.append(response_message.to_dict())
print(messages)

[{'role': 'user', 'content': '서울 날씨는 어떤가요?'}, {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': [], 'tool_calls': [{'id': 'call_m4ESQiAzBF6ezILzJSXTTehI', 'function': {'arguments': '{"location":"Seoul","unit":"celsius"}', 'name': 'get_current_weather'}, 'type': 'function'}]}]


In [5]:
        # "tool_calls": [
        #   {
        #     "id": "call_Ywp8kgOjxJWP243SNEnrXBmx",
        #     "function": {
        #       "arguments": "{\"location\":\"Seoul, South Korea\",\"unit\":\"celsius\"}",
        #       "name": "get_current_weather"
        #     },
        #     "type": "function"


available_functions = {
    "get_current_weather": get_current_weather,
}

# 사용하고 싶은 함수는 여러 개일 수 있으므로 반복문 사용
for tool_call in response_message.tool_calls:
    # 함수를 실행
    function_name = tool_call.function.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.function.arguments)
    function_response = function_to_call(
        location=function_args.get("location"),
        unit=function_args.get("unit"),
    )
    print(function_response)

    # 함수 실행 결과를 대화 이력으로 messages에 추가
    messages.append(
        {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": function_response
        }
    )

{"location": "Seoul", "temperature": "10", "unit": "celsius"}


In [ ]:
# 이 시점의 messages를 표시해 보자.
print(json.dumps(messages, ensure_ascii=False, indent=2))

[
  {
    "role": "user",
    "content": "서울 날씨는 어떤가요?"
  },
  {
    "content": null,
    "refusal": null,
    "role": "assistant",
    "annotations": [],
    "tool_calls": [
      {
        "id": "call_m4ESQiAzBF6ezILzJSXTTehI",
        "function": {
          "arguments": "{\"location\":\"Seoul\",\"unit\":\"celsius\"}",
          "name": "get_current_weather"
        },
        "type": "function"
      }
    ]
  },
  {
    "tool_call_id": "call_m4ESQiAzBF6ezILzJSXTTehI",
    "role": "tool",
    "name": "get_current_weather",
    "content": "{\"location\": \"Seoul\", \"temperature\": \"10\", \"unit\": \"celsius\"}"
  }
]


In [7]:
# 이 messages를 사용해 다시 한번 Chat Completions API에 요청을 보낸다.
second_response = client.chat.completions.create(
    model="gpt-6-luna",
    messages=messages,
)
print(second_response.to_json(indent=2))

{
  "id": "chatcmpl-Ed9noUdS9GoFZNnWWqonD86bF2p4x",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "서울은 현재 **10°C**입니다.",
        "refusal": null,
        "role": "assistant",
        "annotations": []
      }
    }
  ],
  "created": 1793001180,
  "model": "gpt-6-luna",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 12,
    "prompt_tokens": 70,
    "total_tokens": 82,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cache_write_tokens": 0,
      "cached_tokens": 0
    }
  }
}
